# Notebook 4 — Individual Model Training & Tuning
## HMDA 2023 Loan Denial Prediction

---

In [13]:
# ============================================================
# Imports, SparkSession & MLflow Setup
# ============================================================
# ─── Core Spark ───
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, ArrayType
from pyspark.ml.linalg import Vectors, VectorUDT, DenseVector

# ─── PySpark ML: All Available Classifiers ───
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    NaiveBayes,
    LinearSVC,
)

# ─── PySpark ML: Evaluation & Tuning ───
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.tuning import (
    ParamGridBuilder,
    TrainValidationSplit,
    CrossValidator,
)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

# ─── Visualization & Utilities ───
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json, time, os, gc, warnings
warnings.filterwarnings("ignore")

# ─── MLflow for experiment tracking ───
try:
    import mlflow
    import mlflow.spark
    MLFLOW_AVAILABLE = True
    print("MLflow imported successfully")
except ImportError:
    MLFLOW_AVAILABLE = False
    print("MLflow not installed — continuing without tracking.")

# ─── Spark Session ───
spark = (SparkSession.builder
    .appName("HMDA_2023_ModelTraining_4")
    .master("local[2]")                     
    .config("spark.driver.memory", "6g")    
    .config("spark.driver.maxResultSize", "2g") 
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.driver.extraJavaOptions",
            "-XX:+UseG1GC -XX:G1HeapRegionSize=16m "
            "-XX:InitiatingHeapOccupancyPercent=35")
    .config("spark.memory.fraction", "0.6")
    .config("spark.memory.storageFraction", "0.3")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | Cores: {spark.sparkContext.defaultParallelism}")

MLflow not installed — continuing without tracking.
Spark 3.5.5 | Cores: 2


In [14]:
# ============================================================
# Load Data & Initialize MLflow Experiment
# ============================================================

DATA_DIR  = "../data/processed"
MODEL_DIR = f"{DATA_DIR}/models"
os.makedirs(MODEL_DIR, exist_ok=True)

# Load feature-engineered Parquet
train_df = spark.read.parquet(f"{DATA_DIR}/train_features.parquet").cache()
test_df  = spark.read.parquet(f"{DATA_DIR}/test_features.parquet").cache()

train_count = train_df.count()
test_count  = test_df.count()
feature_dim = train_df.first()["features"].size

# Load metadata
with open(f"{DATA_DIR}/feature_metadata.json") as f:
    feature_meta = json.load(f)

print("=" * 70)
print("DATA LOADED")
print("=" * 70)
print(f"  Train: {train_count:,} rows x {feature_dim} features")
print(f"  Test:  {test_count:,} rows x {feature_dim} features")

# Label distribution
for name, df in [("Train", train_df), ("Test", test_df)]:
    counts = df.groupBy("label").count().orderBy("label").collect()
    total = sum(r["count"] for r in counts)
    print(f"  {name}: " + " | ".join(
        f"{'Originated' if r['label']==0 else 'Denied'}: {r['count']:,} ({r['count']/total*100:.1f}%)"
        for r in counts))


print("\nMLflow not available — metrics will be stored in-memory only.")

26/03/02 00:44:56 WARN MemoryStore: Not enough space to cache rdd_757_2 in memory! (computed 708.1 MiB so far)
26/03/02 00:44:56 WARN BlockManager: Persisting block rdd_757_2 to disk instead.
26/03/02 00:44:56 WARN MemoryStore: Not enough space to cache rdd_757_3 in memory! (computed 708.1 MiB so far)
26/03/02 00:44:56 WARN BlockManager: Persisting block rdd_757_3 to disk instead.
26/03/02 00:44:58 WARN MemoryStore: Not enough space to cache rdd_757_3 in memory! (computed 708.1 MiB so far)
26/03/02 00:44:58 WARN MemoryStore: Not enough space to cache rdd_757_3 in memory! (computed 708.1 MiB so far)


DATA LOADED
  Train: 6,148,985 rows x 145 features
  Test:  1,533,558 rows x 145 features


26/03/02 00:45:01 WARN MemoryStore: Not enough space to cache rdd_757_1 in memory! (computed 708.1 MiB so far)
26/03/02 00:45:02 WARN MemoryStore: Not enough space to cache rdd_757_3 in memory! (computed 708.1 MiB so far)


  Train: Originated: 4,553,462 (74.1%) | Denied: 1,595,523 (25.9%)
  Test: Originated: 1,136,116 (74.1%) | Denied: 397,442 (25.9%)

MLflow not available — metrics will be stored in-memory only.


## Section 1 — Class Imbalance: The Foundation of Everything

In [15]:
# ============================================================
# Compute Class Weights & Prepare Weighted DataFrames
# ============================================================

label_counts = {row["label"]: row["count"]
                for row in train_df.groupBy("label").count().collect()}
n_orig   = label_counts[0.0]
n_denied = label_counts[1.0]
total    = n_orig + n_denied

w0 = total / (2.0 * n_orig)     # originated weight
w1 = total / (2.0 * n_denied)   # denied weight (higher)

print(f"Class weights: originated={w0:.4f}, denied={w1:.4f} (ratio {w1/w0:.1f}x)")

# Add weight column
train_w = train_df.withColumn(
    "weight", F.when(F.col("label") == 1.0, F.lit(w1)).otherwise(F.lit(w0))
).cache()
test_w = test_df.withColumn(
    "weight", F.when(F.col("label") == 1.0, F.lit(w1)).otherwise(F.lit(w0))
).cache()

# Unpersist the raw DFs — we only need the weighted versions from here
train_df.unpersist()
test_df.unpersist()

print("Weighted DataFrames cached. Raw DataFrames unpersisted to free memory.")
print(f"  train_w: {train_w.count():,} rows")
print(f"  test_w:  {test_w.count():,} rows")

train_sample = train_w.sample(fraction=0.1, seed=42).cache()
sample_count = train_sample.count()
print(f"\n  train_sample (10% for tuning): {sample_count:,} rows")

26/03/02 00:45:02 WARN MemoryStore: Not enough space to cache rdd_757_1 in memory! (computed 708.1 MiB so far)


Class weights: originated=0.6752, denied=1.9269 (ratio 2.9x)
Weighted DataFrames cached. Raw DataFrames unpersisted to free memory.


26/03/02 00:45:03 WARN CacheManager: Asked to cache already cached data.
26/03/02 00:45:03 WARN CacheManager: Asked to cache already cached data.
26/03/02 00:45:04 WARN MemoryStore: Not enough space to cache rdd_74_2 in memory! (computed 708.1 MiB so far)


  train_w: 6,148,985 rows
  test_w:  1,533,558 rows

  train_sample (10% for tuning): 614,985 rows


26/03/02 00:45:04 WARN CacheManager: Asked to cache already cached data.


## Section 2 — Evaluation Framework & MLflow Integration

In [16]:
# ============================================================
# Shared Evaluation Utilities + MLflow Logger
# ============================================================

# ── PySpark Evaluators ──
eval_roc  = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
eval_pr   = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR")
eval_f1   = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
eval_acc  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# ── Global storage ──
ALL_RESULTS = {}
ALL_PREDICTIONS = {} 

# ── Storage for best hyperparams
BEST_PARAMS = {}


def evaluate_model(predictions, model_name, train_time=0.0, params=None,
                   tags=None, log_mlflow=True):
    """Evaluate a model, print results, store globally, and log to MLflow."""
    metrics = {}
    try:
        metrics["ROC-AUC"] = eval_roc.evaluate(predictions)
    except:
        metrics["ROC-AUC"] = 0.5
    try:
        metrics["PR-AUC"] = eval_pr.evaluate(predictions)
    except:
        metrics["PR-AUC"] = 0.0
    metrics["F1"]       = eval_f1.evaluate(predictions)
    metrics["Accuracy"] = eval_acc.evaluate(predictions)

    # Confusion matrix — compute in ONE pass
    cm = (predictions
        .select(
            F.col("label").cast("int").alias("actual"),
            F.col("prediction").cast("int").alias("pred"))
        .groupBy("actual", "pred").count()
        .collect()
    )
    cm_dict = {(r["actual"], r["pred"]): r["count"] for r in cm}
    tp = cm_dict.get((1, 1), 0)
    fp = cm_dict.get((0, 1), 0)
    fn = cm_dict.get((1, 0), 0)
    tn = cm_dict.get((0, 0), 0)

    d_prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    d_rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
    d_f1   = 2*d_prec*d_rec/(d_prec+d_rec) if (d_prec+d_rec)>0 else 0.0

    metrics.update({
        "Denial_Precision": d_prec, "Denial_Recall": d_rec, "Denial_F1": d_f1,
        "Train_Time_s": train_time,
        "Confusion": {"TP": tp, "FP": fp, "FN": fn, "TN": tn}
    })

    ALL_RESULTS[model_name] = metrics

    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    print(f"  ROC-AUC:          {metrics['ROC-AUC']:.4f}")
    print(f"  PR-AUC:           {metrics['PR-AUC']:.4f}  *** primary metric")
    print(f"  F1 (weighted):    {metrics['F1']:.4f}")
    print(f"  Accuracy:         {metrics['Accuracy']:.4f}")
    print(f"  ------------------------------------")
    print(f"  Denial Precision: {d_prec:.4f}")
    print(f"  Denial Recall:    {d_rec:.4f}")
    print(f"  Denial F1:        {d_f1:.4f}")
    print(f"  ------------------------------------")
    print(f"  Confusion:  TN={tn:,}  FP={fp:,}")
    print(f"              FN={fn:,}  TP={tp:,}")
    print(f"  Time: {train_time:.1f}s")

    if log_mlflow and MLFLOW_AVAILABLE:
        with mlflow.start_run(run_name=model_name):
            if params:
                for k, v in params.items():
                    mlflow.log_param(k, v)
            for k, v in metrics.items():
                if k != "Confusion" and isinstance(v, (int, float)):
                    mlflow.log_metric(k.replace("-", "_"), v)
            if tags:
                for k, v in tags.items():
                    mlflow.set_tag(k, v)
            mlflow.set_tag("model_name", model_name)

    return metrics


def store_predictions_for_ensemble(predictions, model_name, sample_frac=0.05):
    """Sample predictions to pandas for ensemble diversity analysis."""
    pdf = (predictions
        .select("label", extract_p1("probability").alias("prob"))
        .sample(fraction=sample_frac, seed=42)
        .toPandas()
    )
    ALL_PREDICTIONS[model_name] = pdf
    print(f"  Stored {len(pdf):,} sampled predictions for ensemble analysis")


# Helper UDF
extract_p1 = F.udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.5, DoubleType())

print("Evaluation framework ready.")
print(f"MLflow logging: {'ENABLED' if MLFLOW_AVAILABLE else 'DISABLED'}")

Evaluation framework ready.
MLflow logging: DISABLED


## Model 0 — Majority-Class Baseline

In [17]:
# ============================================================
# Model 0 — Majority-Class Baseline
# ============================================================

start = time.time()

baseline_preds = test_w.select("label").withColumn(
    "prediction", F.lit(0.0)
).withColumn(
    "rawPrediction", F.udf(lambda: Vectors.dense([1.0, 0.0]), VectorUDT())()
).withColumn(
    "probability", F.udf(lambda: Vectors.dense([1.0, 0.0]), VectorUDT())()
)

evaluate_model(baseline_preds, "M0_Baseline", time.time()-start,
               params={"strategy": "majority_class"},
               tags={"family": "baseline", "complexity": "none"})


  M0_Baseline
  ROC-AUC:          0.5000
  PR-AUC:           0.2592  *** primary metric
  F1 (weighted):    0.6305
  Accuracy:         0.7408
  ------------------------------------
  Denial Precision: 0.0000
  Denial Recall:    0.0000
  Denial F1:        0.0000
  ------------------------------------
  Confusion:  TN=1,136,116  FP=0
              FN=397,442  TP=0
  Time: 0.0s


{'ROC-AUC': 0.5,
 'PR-AUC': 0.25916333128580726,
 'F1': 0.6305461960620614,
 'Accuracy': 0.7408366687141927,
 'Denial_Precision': 0.0,
 'Denial_Recall': 0.0,
 'Denial_F1': 0.0,
 'Train_Time_s': 0.02030801773071289,
 'Confusion': {'TP': 0, 'FP': 0, 'FN': 397442, 'TN': 1136116}}

## Model 1 — Naive Bayes

In [18]:
# ============================================================
# Model 1 — Naive Bayes
# ============================================================

print("=" * 70)
print("MODEL 1: NAIVE BAYES")
print("=" * 70)

start = time.time()

nb = NaiveBayes(
    featuresCol="features", labelCol="label", weightCol="weight",
    modelType="gaussian", smoothing=1.0,
)

nb_grid = (ParamGridBuilder()
    .addGrid(nb.smoothing, [0.1, 0.5, 1.0, 2.0])
    .build()
)

nb_tvs = TrainValidationSplit(
    estimator=nb, estimatorParamMaps=nb_grid,
    evaluator=eval_pr, trainRatio=0.8, seed=42,
    parallelism=1, 
)

print(f"Tuning {len(nb_grid)} smoothing values...")
nb_tvs_model = nb_tvs.fit(train_w)
nb_time = time.time() - start

best_nb = nb_tvs_model.bestModel
best_smoothing = best_nb._java_obj.getSmoothing()
BEST_PARAMS["NB"] = {"smoothing": best_smoothing}

print(f"  Best smoothing: {best_smoothing}")
for p, s in zip(nb_grid, nb_tvs_model.validationMetrics):
    print(f"    smoothing={p[nb.smoothing]:.1f} -> {s:.4f}")

nb_preds = nb_tvs_model.transform(test_w)
evaluate_model(nb_preds, "M1_NaiveBayes", nb_time,
    params={"smoothing": best_smoothing, "modelType": "gaussian"},
    tags={"family": "probabilistic", "complexity": "very_low"})

store_predictions_for_ensemble(nb_preds, "NaiveBayes")
nb_preds.unpersist()

del nb_tvs, nb_tvs_model
gc.collect()
print("  Freed TVS objects.")

MODEL 1: NAIVE BAYES
Tuning 4 smoothing values...


26/03/02 00:45:14 WARN MemoryStore: Not enough space to cache rdd_924_1 in memory! (computed 708.2 MiB so far)
26/03/02 00:45:14 WARN BlockManager: Persisting block rdd_924_1 to disk instead.
26/03/02 00:45:14 WARN MemoryStore: Not enough space to cache rdd_924_2 in memory! (computed 260.1 MiB so far)
26/03/02 00:45:14 WARN BlockManager: Persisting block rdd_924_2 to disk instead.
26/03/02 00:45:17 WARN MemoryStore: Not enough space to cache rdd_924_3 in memory! (computed 708.2 MiB so far)
26/03/02 00:45:17 WARN BlockManager: Persisting block rdd_924_3 to disk instead.
26/03/02 00:45:23 WARN MemoryStore: Not enough space to cache rdd_74_2 in memory! (computed 708.1 MiB so far)
26/03/02 00:45:52 WARN MemoryStore: Not enough space to cache rdd_74_3 in memory! (computed 452.1 MiB so far)


  Best smoothing: 1.0
    smoothing=0.1 -> 0.9937
    smoothing=0.5 -> 0.9937
    smoothing=1.0 -> 0.9937
    smoothing=2.0 -> 0.9937



  M1_NaiveBayes
  ROC-AUC:          0.9986
  PR-AUC:           0.9937  *** primary metric
  F1 (weighted):    0.9917
  Accuracy:         0.9917
  ------------------------------------
  Denial Precision: 0.9930
  Denial Recall:    0.9748
  Denial F1:        0.9838
  ------------------------------------
  Confusion:  TN=1,133,404  FP=2,712
              FN=10,023  TP=387,419
  Time: 40.5s


  Stored 76,364 sampled predictions for ensemble analysis
  Freed TVS objects.


## Model 2 — Logistic Regression

In [19]:
# ============================================================
#  Model 2 — Logistic Regression with Tuning
# ============================================================

print("=" * 70)
print("MODEL 2: LOGISTIC REGRESSION")
print("=" * 70)

start = time.time()

lr = LogisticRegression(
    featuresCol="features", labelCol="label", weightCol="weight",
    maxIter=100, family="binomial", standardization=False,
)

lr_grid = (ParamGridBuilder()
    .addGrid(lr.regParam, [0.001, 0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.3, 0.7, 1.0])
    .build()
)

lr_tvs = TrainValidationSplit(
    estimator=lr, estimatorParamMaps=lr_grid,
    evaluator=eval_pr, trainRatio=0.8, seed=42,
    parallelism=1,
)

print(f"Tuning {len(lr_grid)} combinations: regParam(3) x elasticNet(4)")
lr_tvs_model = lr_tvs.fit(train_w)
lr_time = time.time() - start

best_lr = lr_tvs_model.bestModel
best_rp = best_lr._java_obj.getRegParam()
best_en = best_lr._java_obj.getElasticNetParam()
BEST_PARAMS["LR"] = {"regParam": best_rp, "elasticNetParam": best_en}

print(f"\nBest: regParam={best_rp}, elasticNet={best_en}")
scored = sorted(zip(lr_grid, lr_tvs_model.validationMetrics), key=lambda x: -x[1])
for p, s in scored[:5]:
    print(f"  reg={p[lr.regParam]:.3f}, en={p[lr.elasticNetParam]:.1f} -> {s:.4f}")

lr_preds = lr_tvs_model.transform(test_w)
evaluate_model(lr_preds, "M2_LogisticRegression", lr_time,
    params={"regParam": best_rp, "elasticNetParam": best_en, "maxIter": 100},
    tags={"family": "linear", "complexity": "low"})

store_predictions_for_ensemble(lr_preds, "LR")
lr_preds.unpersist()

# Coefficient analysis
coeffs = best_lr.coefficients.toArray()
n_nonzero = sum(1 for c in coeffs if abs(c) > 1e-6)
print(f"\nCoefficients: {len(coeffs)} total, {n_nonzero} non-zero")
print(f"Intercept: {best_lr.intercept:.4f}")
top_idx = np.argsort(np.abs(coeffs))[::-1][:10]
print(f"Top 10 by |weight|:")
for rank, idx in enumerate(top_idx, 1):
    print(f"  {rank}. Feature[{idx}]: {coeffs[idx]:+.4f}")

del lr_tvs, lr_tvs_model
gc.collect()
print("  Freed TVS objects.")

MODEL 2: LOGISTIC REGRESSION
Tuning 12 combinations: regParam(3) x elasticNet(4)


26/03/02 00:46:12 WARN MemoryStore: Not enough space to cache rdd_74_2 in memory! (computed 708.1 MiB so far)
26/03/02 00:46:13 WARN MemoryStore: Not enough space to cache rdd_1177_3 in memory! (computed 708.2 MiB so far)
26/03/02 00:46:13 WARN BlockManager: Persisting block rdd_1177_3 to disk instead.
26/03/02 00:46:57 WARN MemoryStore: Not enough space to cache rdd_74_2 in memory! (computed 708.1 MiB so far)
26/03/02 00:56:09 WARN MemoryStore: Not enough space to cache rdd_74_3 in memory! (computed 452.1 MiB so far)
26/03/02 00:56:11 WARN MemoryStore: Not enough space to cache rdd_2862_1 in memory! (computed 630.5 MiB so far)
26/03/02 00:56:11 WARN BlockManager: Persisting block rdd_2862_1 to disk instead.
26/03/02 00:56:14 WARN MemoryStore: Not enough space to cache rdd_74_3 in memory! (computed 708.1 MiB so far)
26/03/02 00:56:17 WARN MemoryStore: Not enough space to cache rdd_2862_0 in memory! (computed 630.5 MiB so far)



Best: regParam=0.001, elasticNet=0.0
  reg=0.001, en=0.0 -> 0.9988
  reg=0.001, en=0.3 -> 0.9987
  reg=0.001, en=0.7 -> 0.9983
  reg=0.010, en=0.0 -> 0.9980
  reg=0.001, en=1.0 -> 0.9976



  M2_LogisticRegression
  ROC-AUC:          0.9996
  PR-AUC:           0.9988  *** primary metric
  F1 (weighted):    0.9937
  Accuracy:         0.9937
  ------------------------------------
  Denial Precision: 0.9876
  Denial Recall:    0.9880
  Denial F1:        0.9878
  ------------------------------------
  Confusion:  TN=1,131,189  FP=4,927
              FN=4,762  TP=392,680
  Time: 642.5s


  Stored 76,364 sampled predictions for ensemble analysis

Coefficients: 145 total, 145 non-zero
Intercept: -2.1006
Top 10 by |weight|:
  1. Feature[16]: +4.5885
  2. Feature[26]: -0.3391
  3. Feature[102]: +0.3184
  4. Feature[105]: -0.2665
  5. Feature[25]: +0.2563
  6. Feature[100]: -0.2130
  7. Feature[107]: +0.2063
  8. Feature[128]: +0.1859
  9. Feature[129]: -0.1859
  10. Feature[2]: -0.1732
  Freed TVS objects.


## Model 3 — Support Vector Machine (Linear SVM)

In [20]:
# ============================================================
# Model 3 — Linear SVM
# ============================================================

print("=" * 70)
print("MODEL 3: LINEAR SVM")
print("=" * 70)

start = time.time()

svm = LinearSVC(
    featuresCol="features", labelCol="label",
    weightCol="weight",
    maxIter=100,
    standardization=False,
)

svm_grid = (ParamGridBuilder()
    .addGrid(svm.regParam, [0.001, 0.01, 0.1, 1.0])
    .build()
)

svm_tvs = TrainValidationSplit(
    estimator=svm, estimatorParamMaps=svm_grid,
    evaluator=eval_pr, trainRatio=0.8, seed=42,
    parallelism=1, 
)

print(f"Tuning {len(svm_grid)} regParam values...")
try:
    svm_tvs_model = svm_tvs.fit(train_w)
    svm_time = time.time() - start

    best_svm = svm_tvs_model.bestModel
    best_svm_rp = best_svm._java_obj.getRegParam()
    BEST_PARAMS["SVM"] = {"regParam": best_svm_rp}
    print(f"Best regParam: {best_svm_rp}")

    svm_preds_raw = svm_tvs_model.transform(test_w)
    sigmoid_udf = F.udf(
        lambda v: Vectors.dense([1.0/(1.0+np.exp(v[0])), 1.0/(1.0+np.exp(-v[0]))]),
        VectorUDT()
    )
    svm_preds = svm_preds_raw.withColumn("probability", sigmoid_udf("rawPrediction"))

    evaluate_model(svm_preds, "M3_LinearSVM", svm_time,
        params={"regParam": best_svm_rp, "maxIter": 100},
        tags={"family": "linear", "complexity": "low"})

    store_predictions_for_ensemble(svm_preds, "SVM")
    svm_preds.unpersist()

    svm_coeffs = best_svm.coefficients.toArray()
    n_sv_nz = sum(1 for c in svm_coeffs if abs(c) > 1e-6)
    print(f"\nSVM coefficients: {len(svm_coeffs)} total, {n_sv_nz} non-zero")

except Exception as e:
    print(f"SVM training failed: {e}")
    print("Skipping SVM and continuing...")
    ALL_RESULTS["M3_LinearSVM"] = {
        "ROC-AUC": 0, "PR-AUC": 0, "F1": 0, "Accuracy": 0,
        "Denial_Precision": 0, "Denial_Recall": 0, "Denial_F1": 0,
        "Train_Time_s": 0, "Confusion": {"TP":0,"FP":0,"FN":0,"TN":0}
    }
finally:
    try: del svm_tvs, svm_tvs_model
    except: pass
    gc.collect()
    print("  Freed TVS objects.")

MODEL 3: LINEAR SVM
Tuning 4 regParam values...


26/03/02 00:57:06 WARN MemoryStore: Not enough space to cache rdd_3023_1 in memory! (computed 708.2 MiB so far)
26/03/02 00:57:06 WARN BlockManager: Persisting block rdd_3023_1 to disk instead.
26/03/02 00:57:06 WARN MemoryStore: Not enough space to cache rdd_3023_0 in memory! (computed 708.1 MiB so far)
26/03/02 00:57:06 WARN BlockManager: Persisting block rdd_3023_0 to disk instead.
26/03/02 00:57:08 WARN MemoryStore: Not enough space to cache rdd_3023_3 in memory! (computed 708.2 MiB so far)
26/03/02 00:57:08 WARN BlockManager: Persisting block rdd_3023_3 to disk instead.
26/03/02 00:57:16 WARN MemoryStore: Not enough space to cache rdd_3023_3 in memory! (computed 708.2 MiB so far)
26/03/02 00:57:44 WARN MemoryStore: Not enough space to cache rdd_74_2 in memory! (computed 708.1 MiB so far)
26/03/02 00:59:30 WARN MemoryStore: Not enough space to cache rdd_74_2 in memory! (computed 708.1 MiB so far)
26/03/02 00:59:30 WARN MemoryStore: Not enough space to cache rdd_74_3 in memory! (com

Best regParam: 0.001



  M3_LinearSVM
  ROC-AUC:          0.9994
  PR-AUC:           0.9977  *** primary metric
  F1 (weighted):    0.9942
  Accuracy:         0.9943
  ------------------------------------
  Denial Precision: 0.9931
  Denial Recall:    0.9846
  Denial F1:        0.9889
  ------------------------------------
  Confusion:  TN=1,133,413  FP=2,703
              FN=6,114  TP=391,328
  Time: 188.1s


  Stored 76,364 sampled predictions for ensemble analysis

SVM coefficients: 145 total, 143 non-zero
  Freed TVS objects.


## Model 4 — Decision Tree

In [21]:
# ============================================================
# Model 4 — Decision Tree
# ============================================================

print("=" * 70)
print("MODEL 4: DECISION TREE")
print("=" * 70)

start = time.time()

dt = DecisionTreeClassifier(
    featuresCol="features", labelCol="label", weightCol="weight",
    impurity="gini", seed=42,
)

dt_grid = (ParamGridBuilder()
    .addGrid(dt.maxDepth, [5, 10, 15, 20])
    .addGrid(dt.minInstancesPerNode, [100, 500, 1000])
    .build()
)

dt_tvs = TrainValidationSplit(
    estimator=dt, estimatorParamMaps=dt_grid,
    evaluator=eval_pr, trainRatio=0.8, seed=42,
    parallelism=1,  # OOM FIX: was 2
)

print(f"Tuning {len(dt_grid)} combos: maxDepth(4) x minInstances(3)")
dt_tvs_model = dt_tvs.fit(train_w)
dt_time = time.time() - start

best_dt = dt_tvs_model.bestModel
BEST_PARAMS["DT"] = {
    "maxDepth": best_dt.depth,
    "minInstancesPerNode": best_dt._java_obj.getMinInstancesPerNode()
}
print(f"\nBest: depth={best_dt.depth}, nodes={best_dt.numNodes}")

dt_preds = dt_tvs_model.transform(test_w)
evaluate_model(dt_preds, "M4_DecisionTree", dt_time,
    params={"maxDepth": best_dt.depth, "numNodes": best_dt.numNodes,
            "minInstancesPerNode": best_dt._java_obj.getMinInstancesPerNode()},
    tags={"family": "tree", "complexity": "medium"})

store_predictions_for_ensemble(dt_preds, "DT")
dt_preds.unpersist()

dt_imp = best_dt.featureImportances.toArray()
top_dt = np.argsort(dt_imp)[::-1][:10]
print(f"\nTop 10 features (Gini decrease):")
for r, i in enumerate(top_dt, 1):
    print(f"  {r}. Feature[{i}] = {dt_imp[i]:.4f}")

# OOM FIX: Release TVS (12 DT candidate models)
del dt_tvs, dt_tvs_model
gc.collect()
print("  Freed TVS objects.")

MODEL 4: DECISION TREE
Tuning 12 combos: maxDepth(4) x minInstances(3)


26/03/02 01:00:37 WARN MemoryStore: Not enough space to cache rdd_4441_1 in memory! (computed 708.2 MiB so far)
26/03/02 01:00:37 WARN BlockManager: Persisting block rdd_4441_1 to disk instead.
26/03/02 01:00:37 WARN MemoryStore: Not enough space to cache rdd_4441_0 in memory! (computed 708.1 MiB so far)
26/03/02 01:00:37 WARN BlockManager: Persisting block rdd_4441_0 to disk instead.
26/03/02 01:00:40 WARN MemoryStore: Not enough space to cache rdd_4441_0 in memory! (computed 708.1 MiB so far)
26/03/02 01:00:55 WARN MemoryStore: Not enough space to cache rdd_4472_1 in memory! (computed 764.8 MiB so far)
26/03/02 01:00:56 WARN MemoryStore: Not enough space to cache rdd_4472_1 in memory! (computed 764.8 MiB so far)
26/03/02 01:00:58 WARN MemoryStore: Not enough space to cache rdd_4472_1 in memory! (computed 764.8 MiB so far)
26/03/02 01:01:00 WARN MemoryStore: Not enough space to cache rdd_4472_1 in memory! (computed 764.8 MiB so far)
26/03/02 01:01:01 WARN MemoryStore: Not enough space


Best: depth=15, nodes=531



  M4_DecisionTree
  ROC-AUC:          0.9994
  PR-AUC:           0.9982  *** primary metric
  F1 (weighted):    0.9938
  Accuracy:         0.9938
  ------------------------------------
  Denial Precision: 0.9864
  Denial Recall:    0.9899
  Denial F1:        0.9881
  ------------------------------------
  Confusion:  TN=1,130,674  FP=5,442
              FN=4,011  TP=393,431
  Time: 437.1s


  Stored 76,364 sampled predictions for ensemble analysis

Top 10 features (Gini decrease):
  1. Feature[16] = 0.9458
  2. Feature[117] = 0.0477
  3. Feature[7] = 0.0020
  4. Feature[100] = 0.0006
  5. Feature[83] = 0.0004
  6. Feature[86] = 0.0004
  7. Feature[125] = 0.0003
  8. Feature[0] = 0.0003
  9. Feature[105] = 0.0002
  10. Feature[23] = 0.0002
  Freed TVS objects.


In [22]:
# ============================================================
# Individual Model Leaderboard
# ============================================================

print("=" * 90)
print("INDIVIDUAL MODEL LEADERBOARD (sorted by PR-AUC)")
print("=" * 90)

rows = []
for name, m in ALL_RESULTS.items():
    rows.append({
        "Model": name,
        "ROC-AUC": m["ROC-AUC"],
        "PR-AUC": m["PR-AUC"],
        "Denial_F1": m["Denial_F1"],
        "Denial_Prec": m["Denial_Precision"],
        "Denial_Rec": m["Denial_Recall"],
        "Accuracy": m["Accuracy"],
        "Time_s": m["Train_Time_s"],
    })

leaderboard = pd.DataFrame(rows).sort_values("PR-AUC", ascending=False)
print(leaderboard.to_string(index=False, float_format="%.4f"))

print(f"\n>>> Current best: {leaderboard.iloc[0]['Model']} "
      f"(PR-AUC = {leaderboard.iloc[0]['PR-AUC']:.4f})")

INDIVIDUAL MODEL LEADERBOARD (sorted by PR-AUC)
                Model  ROC-AUC  PR-AUC  Denial_F1  Denial_Prec  Denial_Rec  Accuracy   Time_s
M2_LogisticRegression   0.9996  0.9988     0.9878       0.9876      0.9880    0.9937 642.5357
      M4_DecisionTree   0.9994  0.9982     0.9881       0.9864      0.9899    0.9938 437.1210
         M3_LinearSVM   0.9994  0.9977     0.9889       0.9931      0.9846    0.9943 188.0969
        M1_NaiveBayes   0.9986  0.9937     0.9838       0.9930      0.9748    0.9917  40.4669
          M0_Baseline   0.5000  0.2592     0.0000       0.0000      0.0000    0.7408   0.0203

>>> Current best: M2_LogisticRegression (PR-AUC = 0.9988)


In [23]:
# ============================================================
# Save Models, Params, Results
# ============================================================

print("=" * 70)
print("SAVING ARTIFACTS")
print("=" * 70)

# ── Save best model objects to disk ──
try:
    best_lr = LogisticRegression(
        featuresCol="features", labelCol="label", weightCol="weight",
        maxIter=100, regParam=BEST_PARAMS["LR"]["regParam"],
        elasticNetParam=BEST_PARAMS["LR"]["elasticNetParam"],
        family="binomial", standardization=False,
    ).fit(train_w)
    best_lr.save(f"{MODEL_DIR}/best_lr")
    # print(f"  Saved: {MODEL_DIR}/best_lr")
except Exception as e:
    print(f"  LR save skipped: {e}")


try:
    best_dt_model = DecisionTreeClassifier(
        featuresCol="features", labelCol="label", weightCol="weight",
        maxDepth=BEST_PARAMS["DT"]["maxDepth"],
        minInstancesPerNode=BEST_PARAMS["DT"]["minInstancesPerNode"],
        impurity="gini", seed=42,
    ).fit(train_w)
    best_dt_model.save(f"{MODEL_DIR}/best_dt")
    # print(f"  Saved: {MODEL_DIR}/best_dt")
    del best_dt_model
except Exception as e:
    print(f"  DT save skipped: {e}")

# ── Save metrics ──
def sanitize(obj):
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, dict): return {k: sanitize(v) for k, v in obj.items()}
    return obj

results_path = f"{DATA_DIR}/model_results_4.json"
with open(results_path, "w") as f:
    json.dump(sanitize(ALL_RESULTS), f, indent=2, default=str)
# print(f"  Saved: {results_path}")

# ── Save best params ──
params_path = f"{DATA_DIR}/best_params_4.json"
with open(params_path, "w") as f:
    json.dump(sanitize(BEST_PARAMS), f, indent=2, default=str)
# print(f"  Saved: {params_path}")

# ── Save ensemble predictions (pandas) ──
preds_path = f"{DATA_DIR}/ensemble_predictions_4.json"
preds_export = {k: v.to_dict(orient="list") for k, v in ALL_PREDICTIONS.items()}
with open(preds_path, "w") as f:
    json.dump(preds_export, f)
# print(f"  Saved: {preds_path}")

# ── Save leaderboard ──
lb_path = f"{DATA_DIR}/leaderboard_4.csv"
leaderboard.to_csv(lb_path, index=False)
# print(f"  Saved: {lb_path}")

SAVING ARTIFACTS


26/03/02 01:08:02 WARN MemoryStore: Not enough space to cache rdd_74_3 in memory! (computed 452.1 MiB so far)
26/03/02 01:08:05 WARN MemoryStore: Not enough space to cache rdd_5677_1 in memory! (computed 630.5 MiB so far)
26/03/02 01:08:05 WARN BlockManager: Persisting block rdd_5677_1 to disk instead.
26/03/02 01:08:08 WARN MemoryStore: Not enough space to cache rdd_74_3 in memory! (computed 708.1 MiB so far)
26/03/02 01:08:10 WARN MemoryStore: Not enough space to cache rdd_5677_0 in memory! (computed 630.5 MiB so far)
26/03/02 01:08:47 WARN MemoryStore: Not enough space to cache rdd_74_2 in memory! (computed 708.1 MiB so far)
26/03/02 01:08:48 WARN MemoryStore: Not enough space to cache rdd_74_2 in memory! (computed 452.1 MiB so far)
26/03/02 01:08:50 WARN MemoryStore: Not enough space to cache rdd_74_2 in memory! (computed 452.1 MiB so far)
26/03/02 01:08:52 WARN MemoryStore: Not enough space to cache rdd_5793_0 in memory! (computed 553.8 MiB so far)
26/03/02 01:08:52 WARN BlockMana

In [24]:
# ============================================================
# Stop Spark — Free JVM
# ============================================================

train_w.unpersist()
test_w.unpersist()
train_sample.unpersist()

spark.stop()
print("Spark stopped. JVM memory fully released.")

Spark stopped. JVM memory fully released.
